# 02. Multi-horizon TP hazard probability TCN

완료된 30봉만으로 보수적 net +3%가 2·3·4·5분에 **최초 도달할 조건부 hazard**를
학습한다. 분별 TP 유지(`at`)는 보조 head로 학습한다.

- 모든 인접 positive를 유지한다.
- class weight, positive oversampling, episode 역가중을 사용하지 않는다.
- 날짜 expanding OOF를 만들 때 내부 마지막 날짜로 epoch만 선택한 뒤 과거 날짜 전체를
  같은 epoch만큼 다시 학습하므로 OOF 날짜 target은 checkpoint 선택에 사용되지 않는다.
- rolling calibration은 해당 OOF 날짜보다 과거에 생성된 OOF 예측만 사용한다.
- 마지막 날짜 Test는 model, calibration, threshold 선택에 사용하지 않는다.

In [1]:
from pathlib import Path
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss, roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


SEED = 42
PREPROCESS_VERSION = 'scalp_30m_ohlcv_net3_multihorizon_5m_v2'
MODEL_VERSION = 'buy_multihorizon_hazard_tcn_v1'
MODEL_HORIZONS = [2, 3, 4, 5]
MAX_EPOCHS = 18
PATIENCE = 4
BATCH_SIZE = 1024
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 1e-3
AT_LOSS_WEIGHT = 0.25
STAKE_USD = 100.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

schema = json.loads((PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_schema.json').read_text())
metadata = pd.read_parquet(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_features.npz') as loaded:
    all_sequence = loaded['sequence'].astype(np.float32)
    all_horizon_targets = loaded['horizon_targets'].astype(np.float32)
    all_at_targets = loaded['at_horizon_targets'].astype(np.float32)
    all_hazard_events = loaded['hazard_events'].astype(np.float32)
    all_hazard_at_risk = loaded['hazard_at_risk'].astype(np.float32)

default_indices = [schema['feature_names'].index(name) for name in schema['default_feature_names']]
sequence = np.ascontiguousarray(all_sequence[:, :, default_indices], dtype=np.float32)
column_indices = np.asarray([horizon - 1 for horizon in MODEL_HORIZONS])
y_by = np.ascontiguousarray(all_horizon_targets[:, column_indices], dtype=np.float32)
y_at = np.ascontiguousarray(all_at_targets[:, column_indices], dtype=np.float32)
y_event = np.ascontiguousarray(all_hazard_events[:, column_indices], dtype=np.float32)
y_risk = np.ascontiguousarray(all_hazard_at_risk[:, column_indices], dtype=np.float32)
del all_sequence, all_horizon_targets, all_at_targets, all_hazard_events, all_hazard_at_risk

sessions = sorted(metadata['session'].unique())
TEST_SESSION = sessions[-1]
OOF_SESSIONS = sessions[-6:-1]
assert len(OOF_SESSIONS) == 5 and all(session < TEST_SESSION for session in OOF_SESSIONS)
assert np.array_equal(metadata['feature_index'].to_numpy(), np.arange(len(metadata)))
assert y_by[:, 0].sum() > 0 and np.all(np.diff(y_by, axis=1) >= 0)
assert np.array_equal(y_event.sum(axis=1), y_by[:, -1])

print('device:', DEVICE)
print('sequence:', sequence.shape, '| horizons:', MODEL_HORIZONS)
print('sessions:', sessions)
print('OOF:', OOF_SESSIONS, '| Test:', TEST_SESSION)
print('cumulative prevalence:', dict(zip(MODEL_HORIZONS, y_by.mean(axis=0))))

device: cuda
sequence: (72181, 30, 38) | horizons: [2, 3, 4, 5]
sessions: ['session_2026-07-07', 'session_2026-07-08', 'session_2026-07-09', 'session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16', 'session_2026-07-17']
OOF: ['session_2026-07-10', 'session_2026-07-13', 'session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16'] | Test: session_2026-07-17
cumulative prevalence: {2: np.float32(0.0057217274), 3: np.float32(0.016333938), 4: np.float32(0.028026765), 5: np.float32(0.039442513)}


## 소형 probability TCN과 proper loss

30봉 안에서 거래량 가속과 가격·VWAP 위치 변화를 잡도록 dilation 1·2·4·8 residual
convolution을 사용한다. hazard loss는 event 발생 전까지의 at-risk 구간에만 자연 비율의
BCE를 적용한다. `at` head도 unweighted BCE이며 보조 loss로만 사용한다.

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def fit_robust_scaler(indices, max_windows=20_000, seed=SEED):
    indices = np.asarray(indices, dtype=np.int64)
    rng = np.random.default_rng(seed)
    if len(indices) > max_windows:
        indices = rng.choice(indices, size=max_windows, replace=False)
    sample = sequence[indices].reshape(-1, sequence.shape[-1])
    center = np.median(sample, axis=0).astype(np.float32)
    q25, q75 = np.percentile(sample, [25, 75], axis=0)
    scale = np.maximum((q75 - q25).astype(np.float32), 1e-4)
    return center, scale


def scaled_sequence(indices, center, scale):
    values = (sequence[np.asarray(indices)] - center[None, None, :]) / scale[None, None, :]
    return np.clip(values, -10, 10).astype(np.float32)


class ResidualTemporalBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.norm = nn.GroupNorm(1, channels)
        self.depthwise = nn.Conv1d(
            channels, channels, kernel_size=3, padding=dilation,
            dilation=dilation, groups=channels,
        )
        self.pointwise = nn.Sequential(
            nn.Conv1d(channels, channels * 2, kernel_size=1), nn.GELU(),
            nn.Dropout(dropout), nn.Conv1d(channels * 2, channels, kernel_size=1),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.depthwise(self.norm(x))
        x = self.pointwise(x)
        return residual + self.dropout(x)


def empirical_logit(values, epsilon=1e-4):
    probability = float(np.clip(np.mean(values), epsilon, 1 - epsilon))
    return math.log(probability / (1 - probability))


class MultiHorizonHazardTCN(nn.Module):
    def __init__(self, input_dim, hazard_bias, at_bias, channels=48, dropout=0.15):
        super().__init__()
        self.input_projection = nn.Conv1d(input_dim, channels, kernel_size=1)
        self.blocks = nn.ModuleList([
            ResidualTemporalBlock(channels, dilation, dropout) for dilation in [1, 2, 4, 8]
        ])
        self.final_norm = nn.GroupNorm(1, channels)
        self.shared = nn.Sequential(
            nn.LayerNorm(channels * 3), nn.Linear(channels * 3, 64),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.hazard_head = nn.Linear(64, len(MODEL_HORIZONS))
        self.at_head = nn.Linear(64, len(MODEL_HORIZONS))
        with torch.no_grad():
            self.hazard_head.bias.copy_(torch.as_tensor(hazard_bias, dtype=torch.float32))
            self.at_head.bias.copy_(torch.as_tensor(at_bias, dtype=torch.float32))

    def forward(self, x):
        x = self.input_projection(x.transpose(1, 2))
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        pooled = torch.cat([x[:, :, -1], x.mean(dim=2), x.amax(dim=2)], dim=1)
        shared = self.shared(pooled)
        return {'hazard_logits': self.hazard_head(shared), 'at_logits': self.at_head(shared)}


def initial_biases(indices):
    hazard_rate = y_event[indices].sum(axis=0) / np.maximum(y_risk[indices].sum(axis=0), 1)
    at_rate = y_at[indices].mean(axis=0)
    hazard_bias = [empirical_logit([value]) for value in hazard_rate]
    at_bias = [empirical_logit([value]) for value in at_rate]
    return np.asarray(hazard_bias, np.float32), np.asarray(at_bias, np.float32)


def probability_loss(output, event, risk, at_target):
    hazard_element = nn.functional.binary_cross_entropy_with_logits(
        output['hazard_logits'], event, reduction='none'
    )
    hazard_loss = (hazard_element * risk).sum() / risk.sum().clamp_min(1)
    at_loss = nn.functional.binary_cross_entropy_with_logits(output['at_logits'], at_target)
    return hazard_loss + AT_LOSS_WEIGHT * at_loss, hazard_loss, at_loss


def cumulative_from_hazard(hazard_probability):
    return 1 - np.cumprod(1 - hazard_probability, axis=1)


def expected_calibration_error(actual, probability, bins=10):
    actual = np.asarray(actual)
    probability = np.asarray(probability)
    edges = np.linspace(0, 1, bins + 1)
    value = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probability >= left) & (probability < right if right < 1 else probability <= right)
        if mask.any():
            value += mask.mean() * abs(actual[mask].mean() - probability[mask].mean())
    return float(value)


def safe_roc_auc(actual, probability):
    return float(roc_auc_score(actual, probability)) if np.unique(actual).size == 2 else float('nan')


def safe_pr_auc(actual, probability):
    return float(average_precision_score(actual, probability)) if actual.sum() > 0 else float('nan')


def probability_metrics(actual_by, cumulative_probability):
    rows = []
    for position, horizon in enumerate(MODEL_HORIZONS):
        actual = actual_by[:, position]
        probability = np.clip(cumulative_probability[:, position], 1e-7, 1 - 1e-7)
        rows.append({
            'horizon': horizon, 'samples': len(actual), 'positives': int(actual.sum()),
            'prevalence': float(actual.mean()), 'mean_probability': float(probability.mean()),
            'pr_auc': safe_pr_auc(actual, probability), 'roc_auc': safe_roc_auc(actual, probability),
            'brier': float(brier_score_loss(actual, probability)),
            'log_loss': float(log_loss(actual, probability, labels=[0, 1])),
            'ece': expected_calibration_error(actual, probability),
        })
    return pd.DataFrame(rows)


def validation_objective(logits, at_logits, indices):
    hazard = expit(logits)
    cumulative = cumulative_from_hazard(hazard)
    probability = np.clip(hazard, 1e-7, 1 - 1e-7)
    event, risk = y_event[indices], y_risk[indices]
    hazard_nll = -(
        event * np.log(probability) + (risk - event) * np.log(1 - probability)
    ).sum() / np.maximum(risk.sum(), 1)
    cumulative_brier = np.mean((cumulative - y_by[indices]) ** 2)
    at_probability = np.clip(expit(at_logits), 1e-7, 1 - 1e-7)
    at_nll = -np.mean(
        y_at[indices] * np.log(at_probability) + (1 - y_at[indices]) * np.log(1 - at_probability)
    )
    return float(hazard_nll + cumulative_brier + AT_LOSS_WEIGHT * at_nll)


probe_hazard, probe_at = initial_biases(np.arange(min(len(metadata), 1000)))
probe = MultiHorizonHazardTCN(sequence.shape[-1], probe_hazard, probe_at)
print('parameters:', sum(parameter.numel() for parameter in probe.parameters()))
del probe

parameters: 50648


## 내부 날짜 early stopping 후 과거 전체 refit

각 OOF fold에서 바로 전 날짜는 epoch 선택에만 사용한다. 선택된 epoch 수로 OOF 날짜보다
과거인 날짜 전체를 새 모델로 재학습하므로 OOF target에 맞춘 checkpoint 선택을 막는다.

In [3]:
def make_loader(indices, center, scale, shuffle, seed):
    x = torch.from_numpy(scaled_sequence(indices, center, scale))
    dataset = TensorDataset(
        x, torch.from_numpy(y_event[indices]), torch.from_numpy(y_risk[indices]),
        torch.from_numpy(y_at[indices]), torch.as_tensor(indices, dtype=torch.long),
    )
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),
        pin_memory=DEVICE.type == 'cuda', num_workers=0,
    )


@torch.no_grad()
def predict_model(model, indices, center, scale):
    loader = make_loader(indices, center, scale, False, SEED)
    model.eval()
    hazard_logits, at_logits, row_indices = [], [], []
    for batch_x, _, _, _, batch_index in loader:
        output = model(batch_x.to(DEVICE, non_blocking=True))
        hazard_logits.append(output['hazard_logits'].cpu().numpy())
        at_logits.append(output['at_logits'].cpu().numpy())
        row_indices.append(batch_index.numpy())
    result = (
        np.concatenate(hazard_logits), np.concatenate(at_logits), np.concatenate(row_indices)
    )
    del loader
    return result


def train_epoch(model, loader, optimizer):
    model.train()
    totals = np.zeros(3, dtype=np.float64)
    count = 0
    for batch_x, batch_event, batch_risk, batch_at, _ in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_event = batch_event.to(DEVICE, non_blocking=True)
        batch_risk = batch_risk.to(DEVICE, non_blocking=True)
        batch_at = batch_at.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = model(batch_x)
        loss, hazard_loss, at_loss = probability_loss(output, batch_event, batch_risk, batch_at)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        totals += np.asarray([loss.item(), hazard_loss.item(), at_loss.item()]) * len(batch_x)
        count += len(batch_x)
    return totals / max(count, 1)


def choose_epoch(inner_train_indices, inner_valid_indices, seed):
    set_seed(seed)
    center, scale = fit_robust_scaler(inner_train_indices, seed=seed)
    train_loader = make_loader(inner_train_indices, center, scale, True, seed)
    hazard_bias, at_bias = initial_biases(inner_train_indices)
    model = MultiHorizonHazardTCN(sequence.shape[-1], hazard_bias, at_bias).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=5e-5)
    best_objective, best_epoch, stale = np.inf, 1, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_values = train_epoch(model, train_loader, optimizer)
        valid_logits, valid_at_logits, _ = predict_model(model, inner_valid_indices, center, scale)
        objective = validation_objective(valid_logits, valid_at_logits, inner_valid_indices)
        history.append({
            'epoch': epoch, 'train_loss': train_values[0],
            'train_hazard_loss': train_values[1], 'train_at_loss': train_values[2],
            'valid_objective': objective,
        })
        if objective < best_objective - 1e-5:
            best_objective, best_epoch, stale = objective, epoch, 0
        else:
            stale += 1
        scheduler.step()
        if stale >= PATIENCE:
            break
    del model, train_loader
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return best_epoch, pd.DataFrame(history)


def fit_fixed_epochs(train_indices, epochs, seed):
    set_seed(seed)
    center, scale = fit_robust_scaler(train_indices, seed=seed)
    loader = make_loader(train_indices, center, scale, True, seed)
    hazard_bias, at_bias = initial_biases(train_indices)
    model = MultiHorizonHazardTCN(sequence.shape[-1], hazard_bias, at_bias).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(int(epochs), 1), eta_min=5e-5
    )
    history = []
    for epoch in range(1, int(epochs) + 1):
        values = train_epoch(model, loader, optimizer)
        history.append({
            'epoch': epoch, 'train_loss': values[0],
            'train_hazard_loss': values[1], 'train_at_loss': values[2],
        })
        scheduler.step()
    del loader
    gc.collect()
    return model, center, scale, pd.DataFrame(history)

## Expanding OOF 예측

In [4]:
oof_frames = []
fold_records = []
history_frames = []
best_epochs = []
started = time.time()

prediction_columns = [
    'sample_id', 'feature_index', 'session', 'symbol', 'decision_bar_timestamp',
    'decision_available_timestamp', 'entry_timestamp', 'entry_ask',
    'target_net3_within_5m', 'first_hit_minute', 'timeout_net_return_5m',
    'best_min_bid_net_return_1_5m', 'worst_min_bid_net_return_1_5m',
] + schema['label']['horizon_target_columns'] + schema['label']['at_horizon_target_columns']   + schema['label']['hazard_event_columns'] + schema['label']['hazard_at_risk_columns']

for fold, validation_session in enumerate(OOF_SESSIONS, start=1):
    prior_sessions = [session for session in sessions if session < validation_session]
    inner_validation_session = prior_sessions[-1]
    inner_train_sessions = prior_sessions[:-1]
    inner_train_indices = np.flatnonzero(metadata['session'].isin(inner_train_sessions).to_numpy())
    inner_valid_indices = np.flatnonzero(metadata['session'].eq(inner_validation_session).to_numpy())
    refit_indices = np.flatnonzero(metadata['session'].isin(prior_sessions).to_numpy())
    validation_indices = np.flatnonzero(metadata['session'].eq(validation_session).to_numpy())
    seed = SEED + fold

    best_epoch, inner_history = choose_epoch(inner_train_indices, inner_valid_indices, seed)
    best_epochs.append(best_epoch)
    inner_history['fold'] = fold
    inner_history['stage'] = 'inner_epoch_selection'
    inner_history['validation_session'] = validation_session
    history_frames.append(inner_history)

    model, center, scale, refit_history = fit_fixed_epochs(refit_indices, best_epoch, seed + 100)
    refit_history['fold'] = fold
    refit_history['stage'] = 'past_full_refit'
    refit_history['validation_session'] = validation_session
    history_frames.append(refit_history)
    logits, at_logits, predicted_indices = predict_model(model, validation_indices, center, scale)
    assert np.array_equal(predicted_indices, validation_indices)
    hazard_probability = expit(logits)
    cumulative_probability = cumulative_from_hazard(hazard_probability)
    at_probability = expit(at_logits)

    frame = metadata.iloc[validation_indices][prediction_columns].copy().reset_index(drop=True)
    for position, horizon in enumerate(MODEL_HORIZONS):
        frame[f'raw_hazard_logit_{horizon}m'] = logits[:, position]
        frame[f'raw_hazard_probability_{horizon}m'] = hazard_probability[:, position]
        frame[f'raw_tp_by_probability_{horizon}m'] = cumulative_probability[:, position]
        frame[f'raw_tp_at_probability_{horizon}m'] = at_probability[:, position]
    frame['fold'] = fold
    oof_frames.append(frame)

    metrics = probability_metrics(y_by[validation_indices], cumulative_probability)
    p5 = metrics.loc[metrics['horizon'].eq(5)].iloc[0]
    fold_records.append({
        'fold': fold, 'validation_session': validation_session,
        'inner_validation_session': inner_validation_session,
        'train_sessions': len(prior_sessions), 'train_samples': len(refit_indices),
        'validation_samples': len(validation_indices), 'best_epoch': best_epoch,
        'p5_prevalence': p5['prevalence'], 'p5_mean_probability': p5['mean_probability'],
        'p5_pr_auc': p5['pr_auc'], 'p5_roc_auc': p5['roc_auc'],
        'p5_brier': p5['brier'], 'p5_ece': p5['ece'],
    })
    print(
        f"fold {fold}/{len(OOF_SESSIONS)} {validation_session} | epoch={best_epoch} | "
        f"P5 {p5['mean_probability']:.3%}/{p5['prevalence']:.3%} | "
        f"AP={p5['pr_auc']:.4f} Brier={p5['brier']:.5f}"
    )
    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

oof = pd.concat(oof_frames, ignore_index=True).sort_values('entry_timestamp').reset_index(drop=True)
fold_metrics = pd.DataFrame(fold_records)
training_history = pd.concat(history_frames, ignore_index=True)
print(f'OOF elapsed: {(time.time() - started) / 60:.2f} min')
display(fold_metrics)

fold 1/5 session_2026-07-10 | epoch=16 | P5 4.640%/5.220% | AP=0.1245 Brier=0.04743


fold 2/5 session_2026-07-13 | epoch=11 | P5 4.097%/5.092% | AP=0.1573 Brier=0.04518


fold 3/5 session_2026-07-14 | epoch=8 | P5 4.224%/3.777% | AP=0.1200 Brier=0.03422


fold 4/5 session_2026-07-15 | epoch=6 | P5 4.523%/4.280% | AP=0.1303 Brier=0.03890


fold 5/5 session_2026-07-16 | epoch=10 | P5 3.515%/4.005% | AP=0.0859 Brier=0.03762
OOF elapsed: 1.39 min


,fold,validation_session,inner_validation_session,train_sessions,train_samples,validation_samples,best_epoch,p5_prevalence,p5_mean_probability,p5_pr_auc,p5_roc_auc,p5_brier,p5_ece
0,1,session_2026-07-10,session_2026-07-09,3,24872,10230,16,0.052199,0.046399,0.124472,0.762691,0.047429,0.006458
1,2,session_2026-07-13,session_2026-07-10,4,35102,7953,11,0.050924,0.040968,0.157301,0.813317,0.045177,0.010083
2,3,session_2026-07-14,session_2026-07-13,5,43055,7043,8,0.037768,0.042245,0.120004,0.836668,0.034222,0.004603
3,4,session_2026-07-15,session_2026-07-14,6,50098,9556,6,0.042800,0.045226,0.130296,0.791206,0.038901,0.002426
4,5,session_2026-07-16,session_2026-07-15,7,59654,5918,10,0.040047,0.035152,0.085944,0.739814,0.037621,0.011756


## 과거 OOF만 사용하는 rolling Platt calibration

각 hazard logit을 독립적으로 보정한 뒤 누적 확률을 다시 계산한다. 첫 OOF 날짜는 과거
OOF가 없으므로 raw 확률을 유지하고 threshold 선택에서는 제외한다.

In [5]:
def fit_platt_calibrators(frame):
    calibrators = {}
    for horizon in MODEL_HORIZONS:
        risk = frame[f'hazard_at_risk_{horizon}m'].to_numpy(bool)
        x = frame.loc[risk, f'raw_hazard_logit_{horizon}m'].to_numpy().reshape(-1, 1)
        y = frame.loc[risk, f'hazard_event_{horizon}m'].to_numpy(int)
        if len(y) < 100 or np.unique(y).size < 2:
            calibrators[horizon] = {'coef': 1.0, 'intercept': 0.0, 'identity': True}
            continue
        estimator = LogisticRegression(C=10.0, solver='lbfgs', max_iter=1000)
        estimator.fit(x, y)
        calibrators[horizon] = {
            'coef': float(estimator.coef_[0, 0]),
            'intercept': float(estimator.intercept_[0]), 'identity': False,
        }
    return calibrators


def apply_platt(frame, calibrators, prefix='calibrated'):
    result = frame.copy()
    hazard = []
    for horizon in MODEL_HORIZONS:
        parameters = calibrators[horizon]
        logit = result[f'raw_hazard_logit_{horizon}m'].to_numpy()
        probability = expit(parameters['coef'] * logit + parameters['intercept'])
        result[f'{prefix}_hazard_probability_{horizon}m'] = probability
        hazard.append(probability)
    cumulative = cumulative_from_hazard(np.column_stack(hazard))
    for position, horizon in enumerate(MODEL_HORIZONS):
        result[f'{prefix}_tp_by_probability_{horizon}m'] = cumulative[:, position]
    return result


rolling_frames = []
for position, validation_session in enumerate(OOF_SESSIONS):
    current = oof[oof['session'].eq(validation_session)].copy()
    prior = oof[oof['session'].isin(OOF_SESSIONS[:position])]
    if prior.empty:
        calibrators = {
            horizon: {'coef': 1.0, 'intercept': 0.0, 'identity': True}
            for horizon in MODEL_HORIZONS
        }
    else:
        calibrators = fit_platt_calibrators(prior)
    current = apply_platt(current, calibrators, prefix='rolling_calibrated')
    current['calibration_history_sessions'] = position
    current['calibration_history_rows'] = len(prior)
    rolling_frames.append(current)

oof = pd.concat(rolling_frames, ignore_index=True).sort_values('entry_timestamp').reset_index(drop=True)
calibration_oof = oof[oof['calibration_history_sessions'].gt(0)].copy()
raw_columns = [f'raw_tp_by_probability_{horizon}m' for horizon in MODEL_HORIZONS]
calibrated_columns = [f'rolling_calibrated_tp_by_probability_{horizon}m' for horizon in MODEL_HORIZONS]
raw_oof_metrics = probability_metrics(
    calibration_oof[[f'target_net3_by_{horizon}m' for horizon in MODEL_HORIZONS]].to_numpy(),
    calibration_oof[raw_columns].to_numpy(),
)
raw_oof_metrics['probability_type'] = 'raw'
calibrated_oof_metrics = probability_metrics(
    calibration_oof[[f'target_net3_by_{horizon}m' for horizon in MODEL_HORIZONS]].to_numpy(),
    calibration_oof[calibrated_columns].to_numpy(),
)
calibrated_oof_metrics['probability_type'] = 'rolling_calibrated'
oof_probability_metrics = pd.concat([raw_oof_metrics, calibrated_oof_metrics], ignore_index=True)
p5_method_comparison = oof_probability_metrics[oof_probability_metrics['horizon'].eq(5)].sort_values(
    ['brier', 'log_loss']
)
SELECTED_PROBABILITY_TYPE = str(p5_method_comparison.iloc[0]['probability_type'])
FINAL_CALIBRATORS = fit_platt_calibrators(oof)
print('selected probability type from OOF:', SELECTED_PROBABILITY_TYPE)
display(oof_probability_metrics)

selected probability type from OOF: raw


,horizon,samples,positives,prevalence,mean_probability,pr_auc,roc_auc,brier,log_loss,ece,probability_type
0,2,30470,199,0.006531,0.006096,0.028545,0.850924,0.006400,0.033636,0.000435,raw
1,3,30470,557,0.018280,0.017276,0.064702,0.825907,0.017405,0.077975,0.001466,raw
2,4,30470,951,0.031211,0.029513,0.097811,0.807239,0.028955,0.119435,0.001698,raw
3,5,30470,1317,0.043223,0.041469,0.125924,0.796020,0.039209,0.153484,0.001828,raw
4,2,30470,199,0.006531,0.007823,0.028636,0.850412,0.006404,0.033757,0.001292,rolling_calibrated
5,3,30470,557,0.018280,0.020843,0.063807,0.823946,0.017420,0.078347,0.002563,rolling_calibrated
6,4,30470,951,0.031211,0.033991,0.096376,0.805132,0.028974,0.119852,0.002780,rolling_calibrated
7,5,30470,1317,0.043223,0.046512,0.124458,0.794047,0.039239,0.154028,0.003289,rolling_calibrated


## OOF 절대 확률 threshold

calibration history가 존재하는 OOF 날짜만 사용한다. threshold 미달이면 매수하지 않으며,
모든 후보가 손실이어도 양수인 것처럼 대체하지 않는다. 같은 종목을 보유 중일 때 발생한
추가 신호는 sequential 평가에서 제외한다.

In [6]:
def sequential_trades(frame, signal_column):
    candidates = frame[frame[signal_column]].sort_values('entry_timestamp').copy()
    selected = []
    for _, group in candidates.groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.sort_values('entry_timestamp').iterrows():
            if available_at is not None and row['entry_timestamp'] < available_at:
                continue
            selected.append(index)
            hold_minutes = int(row['first_hit_minute']) if row['target_net3_within_5m'] else 5
            available_at = row['entry_timestamp'] + pd.Timedelta(minutes=hold_minutes)
    trades = candidates.loc[selected].sort_values('entry_timestamp').copy()
    if len(trades):
        trades['realized_policy_return'] = np.where(
            trades['target_net3_within_5m'].eq(1), 0.03, trades['timeout_net_return_5m']
        )
    else:
        trades['realized_policy_return'] = pd.Series(dtype=float)
    return trades


def policy_metrics(frame, signal_column):
    trades = sequential_trades(frame, signal_column)
    if trades.empty:
        return {
            'trades': 0, 'sessions_traded': 0, 'tp_precision': float('nan'),
            'win_rate': float('nan'), 'mean_return': float('nan'),
            'median_return': float('nan'), 'risk_adjusted_return': float('nan'),
            'total_pnl_usd': 0.0, 'profit_factor': float('nan'),
            'max_drawdown_usd': 0.0,
        }
    returns = trades['realized_policy_return'].to_numpy()
    pnl = returns * STAKE_USD
    equity = np.cumsum(pnl)
    running_peak = np.maximum.accumulate(np.r_[0.0, equity])
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        'trades': int(len(trades)), 'sessions_traded': int(trades['session'].nunique()),
        'tp_precision': float(trades['target_net3_within_5m'].mean()),
        'win_rate': float((returns > 0).mean()), 'mean_return': float(returns.mean()),
        'median_return': float(np.median(returns)),
        'risk_adjusted_return': float(returns.mean() - returns.std(ddof=1) / math.sqrt(len(returns))) if len(returns) > 1 else float(returns.mean()),
        'total_pnl_usd': float(pnl.sum()),
        'profit_factor': float(gross_profit / gross_loss) if gross_loss > 0 else float('inf'),
        'max_drawdown_usd': float((np.r_[0.0, equity] - running_peak).min()),
    }


threshold_frame = calibration_oof.copy()
p5_column = (
    'raw_tp_by_probability_5m' if SELECTED_PROBABILITY_TYPE == 'raw'
    else 'rolling_calibrated_tp_by_probability_5m'
)
fixed_thresholds = np.r_[np.arange(0.01, 0.101, 0.005), [0.125, 0.15, 0.20, 0.30, 0.40, 0.50]]
quantile_thresholds = threshold_frame[p5_column].quantile(
    [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.925, 0.95, 0.975, 0.99]
).to_numpy()
thresholds = np.unique(np.round(np.r_[fixed_thresholds, quantile_thresholds], 6))
threshold_rows = []
for threshold in thresholds:
    threshold_frame['candidate_signal'] = threshold_frame[p5_column].ge(threshold)
    threshold_rows.append({'probability_threshold': float(threshold), **policy_metrics(threshold_frame, 'candidate_signal')})
threshold_candidates = pd.DataFrame(threshold_rows)
eligible_candidates = threshold_candidates[
    threshold_candidates['trades'].ge(50) & threshold_candidates['sessions_traded'].ge(3)
]
if eligible_candidates.empty:
    raise RuntimeError('최소 OOF 거래 수를 만족하는 threshold가 없습니다.')
selected_policy = eligible_candidates.sort_values(
    ['risk_adjusted_return', 'total_pnl_usd', 'probability_threshold']
).iloc[-1]
SELECTED_THRESHOLD = float(selected_policy['probability_threshold'])
OOF_ECONOMIC_ELIGIBLE = bool(
    selected_policy['risk_adjusted_return'] > 0 and selected_policy['profit_factor'] > 1
)
threshold_frame['selected_signal'] = threshold_frame[p5_column].ge(SELECTED_THRESHOLD)
selected_oof_trades = sequential_trades(threshold_frame, 'selected_signal')
print('selected threshold:', SELECTED_THRESHOLD)
print('OOF economic eligible:', OOF_ECONOMIC_ELIGIBLE)
display(selected_policy.to_frame('value'))
display(threshold_candidates.sort_values(['risk_adjusted_return', 'total_pnl_usd'], ascending=False).head(20))

selected threshold: 0.01
OOF economic eligible: False


,value
probability_threshold,0.010000
trades,4405.000000
sessions_traded,4.000000
tp_precision,0.063564
win_rate,0.310556
mean_return,-0.010264
median_return,-0.008374
risk_adjusted_return,-0.010709
total_pnl_usd,-4521.355876
profit_factor,0.363369


,probability_threshold,trades,sessions_traded,tp_precision,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd
0,0.010000,4405,4,0.063564,0.310556,-0.010264,-0.008374,-0.010709,-4521.355876,0.363369,-4521.375650
1,0.015000,3976,4,0.071932,0.312626,-0.010661,-0.008760,-0.011145,-4238.972916,0.365921,-4238.972916
2,0.020000,3641,4,0.076627,0.313925,-0.010892,-0.009145,-0.011411,-3965.842735,0.370454,-3965.842735
3,0.022689,3467,4,0.079608,0.323911,-0.011087,-0.009225,-0.011626,-3843.794069,0.371235,-3843.794069
4,0.025000,3336,4,0.084233,0.325540,-0.011210,-0.009162,-0.011769,-3739.758448,0.373539,-3739.758448
5,0.030000,3094,4,0.087589,0.326761,-0.011361,-0.009262,-0.011958,-3515.007204,0.378153,-3515.007204
6,0.035000,2888,4,0.093144,0.331717,-0.011482,-0.009262,-0.012116,-3315.890430,0.388851,-3317.555105
7,0.036526,2833,4,0.093540,0.331098,-0.011579,-0.009388,-0.012219,-3280.289691,0.386340,-3281.954365
8,0.040000,2697,4,0.096033,0.334446,-0.011649,-0.009702,-0.012315,-3141.771010,0.392175,-3143.435684
9,0.045000,2518,4,0.098888,0.337967,-0.011825,-0.009547,-0.012521,-2977.449395,0.392340,-2977.449395


## 과거 전체 refit과 고정 Test

OOF에서 선택된 epoch 중앙값으로 Test 이전 날짜 전체를 학습한다. 최종 OOF Platt 계수와
threshold를 고정한 뒤 마지막 날짜를 한 번 평가한다.

In [7]:
FINAL_EPOCHS = int(np.clip(np.rint(np.median(best_epochs)), 1, MAX_EPOCHS))
final_train_indices = np.flatnonzero(metadata['session'].ne(TEST_SESSION).to_numpy())
test_indices = np.flatnonzero(metadata['session'].eq(TEST_SESSION).to_numpy())
final_model, final_center, final_scale, final_history = fit_fixed_epochs(
    final_train_indices, FINAL_EPOCHS, SEED + 999
)
test_logits, test_at_logits, predicted_test_indices = predict_model(
    final_model, test_indices, final_center, final_scale
)
assert np.array_equal(predicted_test_indices, test_indices)

test_hazard = expit(test_logits)
test_cumulative_raw = cumulative_from_hazard(test_hazard)
test_at_probability = expit(test_at_logits)
test_frame = metadata.iloc[test_indices][prediction_columns].copy().reset_index(drop=True)
for position, horizon in enumerate(MODEL_HORIZONS):
    test_frame[f'raw_hazard_logit_{horizon}m'] = test_logits[:, position]
    test_frame[f'raw_hazard_probability_{horizon}m'] = test_hazard[:, position]
    test_frame[f'raw_tp_by_probability_{horizon}m'] = test_cumulative_raw[:, position]
    test_frame[f'raw_tp_at_probability_{horizon}m'] = test_at_probability[:, position]
test_frame = apply_platt(test_frame, FINAL_CALIBRATORS, prefix='calibrated')

test_raw_metrics = probability_metrics(
    y_by[test_indices], test_frame[raw_columns].to_numpy()
)
test_raw_metrics['probability_type'] = 'raw'
test_calibrated_columns = [f'calibrated_tp_by_probability_{horizon}m' for horizon in MODEL_HORIZONS]
test_calibrated_metrics = probability_metrics(
    y_by[test_indices], test_frame[test_calibrated_columns].to_numpy()
)
test_calibrated_metrics['probability_type'] = 'calibrated'
test_probability_metrics = pd.concat([test_raw_metrics, test_calibrated_metrics], ignore_index=True)

TEST_P5_COLUMN = (
    'raw_tp_by_probability_5m' if SELECTED_PROBABILITY_TYPE == 'raw'
    else 'calibrated_tp_by_probability_5m'
)
test_frame['buy_signal'] = test_frame[TEST_P5_COLUMN].ge(SELECTED_THRESHOLD)
test_trades = sequential_trades(test_frame, 'buy_signal')
test_policy = policy_metrics(test_frame, 'buy_signal')
DEPLOYMENT_ELIGIBLE = bool(
    OOF_ECONOMIC_ELIGIBLE and test_policy['trades'] >= 20
    and test_policy['risk_adjusted_return'] > 0 and test_policy['profit_factor'] > 1
)

def reliability_table(frame, probability_column, target_column, dataset_name, bins=10):
    work = frame[[probability_column, target_column]].copy()
    rank = work[probability_column].rank(method='first')
    work['bin'] = pd.qcut(rank, q=min(bins, len(work)), labels=False, duplicates='drop')
    result = work.groupby('bin').agg(
        samples=(target_column, 'size'), mean_probability=(probability_column, 'mean'),
        actual_rate=(target_column, 'mean'), positives=(target_column, 'sum'),
    ).reset_index()
    result['dataset'] = dataset_name
    return result


oof_reliability = reliability_table(
    threshold_frame, p5_column, 'target_net3_by_5m', 'rolling_oof'
)
test_reliability = reliability_table(
    test_frame, TEST_P5_COLUMN, 'target_net3_by_5m', 'test'
)
reliability = pd.concat([oof_reliability, test_reliability], ignore_index=True)
print('final epochs:', FINAL_EPOCHS)
print('Test policy:', test_policy)
print('deployment eligible:', DEPLOYMENT_ELIGIBLE)
display(test_probability_metrics)
display(reliability)
display(test_trades.head(20))

final epochs: 10
Test policy: {'trades': 686, 'sessions_traded': 1, 'tp_precision': 0.03935860058309038, 'win_rate': 0.28717201166180756, 'mean_return': -0.010149491623957178, 'median_return': -0.008709312416613102, 'risk_adjusted_return': -0.011213092319103885, 'total_pnl_usd': -696.2551254034624, 'profit_factor': 0.33802613203835963, 'max_drawdown_usd': -698.1913084295229}
deployment eligible: False


,horizon,samples,positives,prevalence,mean_probability,pr_auc,roc_auc,brier,log_loss,ece,probability_type
0,2,6609,10,0.001513,0.003706,0.004653,0.715805,0.001558,0.012478,0.002193,raw
1,3,6609,46,0.006960,0.010820,0.020865,0.782630,0.007031,0.039766,0.003860,raw
2,4,6609,87,0.013164,0.019047,0.040435,0.790064,0.013057,0.064829,0.005883,raw
3,5,6609,131,0.019821,0.027012,0.059016,0.788562,0.019349,0.088715,0.007191,raw
4,2,6609,10,0.001513,0.004236,0.004653,0.715805,0.001568,0.012681,0.002722,calibrated
5,3,6609,46,0.006960,0.011914,0.020886,0.782701,0.007060,0.039953,0.004954,calibrated
6,4,6609,87,0.013164,0.020679,0.040442,0.790077,0.013107,0.065119,0.007515,calibrated
7,5,6609,131,0.019821,0.029002,0.058992,0.788448,0.019414,0.089010,0.009181,calibrated


,bin,samples,mean_probability,actual_rate,positives,dataset
0,0,3047,0.001580,0.000000,0,rolling_oof
1,1,3047,0.003063,0.005579,17,rolling_oof
2,2,3047,0.005690,0.006892,21,rolling_oof
3,3,3047,0.010629,0.011487,35,rolling_oof
4,4,3047,0.018066,0.015425,47,rolling_oof
5,5,3047,0.029039,0.027896,85,rolling_oof
6,6,3047,0.045389,0.048901,149,rolling_oof
7,7,3047,0.066618,0.069248,211,rolling_oof
8,8,3047,0.094338,0.100427,306,rolling_oof
9,9,3047,0.140278,0.146373,446,rolling_oof


,sample_id,feature_index,session,symbol,decision_bar_timestamp,decision_available_timestamp,entry_timestamp,entry_ask,target_net3_within_5m,first_hit_minute,...,calibrated_hazard_probability_2m,calibrated_hazard_probability_3m,calibrated_hazard_probability_4m,calibrated_hazard_probability_5m,calibrated_tp_by_probability_2m,calibrated_tp_by_probability_3m,calibrated_tp_by_probability_4m,calibrated_tp_by_probability_5m,buy_signal,realized_policy_return
547,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066119,66119,session_2026-07-17,BIYA,2026-07-17 08:49:00+00:00,2026-07-17 08:50:00+00:00,2026-07-17 08:50:00+00:00,3.9300,0,0,...,0.001131,0.004223,0.005978,0.007909,0.001131,0.005349,0.011295,0.019116,True,0.002482
1947,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00067519,67519,session_2026-07-17,CJMB,2026-07-17 08:49:00+00:00,2026-07-17 08:50:00+00:00,2026-07-17 08:50:00+00:00,1.3100,0,0,...,0.005030,0.018708,0.017508,0.018664,0.005030,0.023644,0.040738,0.058642,True,-0.030681
3138,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00068710,68710,session_2026-07-17,GCTK,2026-07-17 08:49:00+00:00,2026-07-17 08:50:00+00:00,2026-07-17 08:50:00+00:00,0.5374,0,0,...,0.002169,0.007733,0.008084,0.009446,0.002169,0.009885,0.017889,0.027167,True,0.009533
552,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066124,66124,session_2026-07-17,BIYA,2026-07-17 08:54:00+00:00,2026-07-17 08:55:00+00:00,2026-07-17 08:55:00+00:00,3.9800,0,0,...,0.001186,0.004494,0.005914,0.008532,0.001186,0.005675,0.011555,0.019989,True,-0.025187
1952,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00067524,67524,session_2026-07-17,CJMB,2026-07-17 08:54:00+00:00,2026-07-17 08:55:00+00:00,2026-07-17 08:55:00+00:00,1.2800,0,0,...,0.004584,0.019793,0.015540,0.016876,0.004584,0.024287,0.039449,0.055660,True,0.023287
3143,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00068715,68715,session_2026-07-17,GCTK,2026-07-17 08:54:00+00:00,2026-07-17 08:55:00+00:00,2026-07-17 08:55:00+00:00,0.5450,1,3,...,0.002141,0.009306,0.009039,0.010288,0.002141,0.011427,0.020363,0.030441,True,0.030000
3146,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00068718,68718,session_2026-07-17,GCTK,2026-07-17 08:57:00+00:00,2026-07-17 08:58:00+00:00,2026-07-17 08:58:00+00:00,0.5700,0,0,...,0.007360,0.017057,0.018473,0.021280,0.007360,0.024292,0.042316,0.062695,True,-0.018557
1957,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00067529,67529,session_2026-07-17,CJMB,2026-07-17 08:59:00+00:00,2026-07-17 09:00:00+00:00,2026-07-17 09:00:00+00:00,1.3200,0,0,...,0.003681,0.013609,0.014474,0.016122,0.003681,0.017240,0.031464,0.047079,True,-0.045600
557,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00066129,66129,session_2026-07-17,BIYA,2026-07-17 08:59:00+00:00,2026-07-17 09:00:00+00:00,2026-07-17 09:00:00+00:00,3.8900,0,0,...,0.002299,0.009057,0.010050,0.011911,0.002299,0.011336,0.021272,0.032929,True,-0.010346
3151,scalp_30m_ohlcv_net3_multihorizon_5m_v2_00068723,68723,session_2026-07-17,GCTK,2026-07-17 09:02:00+00:00,2026-07-17 09:03:00+00:00,2026-07-17 09:03:00+00:00,0.5600,0,0,...,0.012127,0.017418,0.020208,0.026107,0.012127,0.029333,0.048948,0.073778,True,0.050039


## Artifact 저장

In [8]:
def at_probability_metric_frame(frame, dataset_name):
    rows = []
    for horizon in MODEL_HORIZONS:
        actual = frame[f'target_net3_at_{horizon}m'].to_numpy()
        probability = frame[f'raw_tp_at_probability_{horizon}m'].to_numpy()
        rows.append({
            'dataset': dataset_name, 'horizon': horizon, 'samples': len(actual),
            'positives': int(actual.sum()), 'prevalence': float(actual.mean()),
            'mean_probability': float(probability.mean()),
            'pr_auc': safe_pr_auc(actual, probability),
            'roc_auc': safe_roc_auc(actual, probability),
            'brier': float(brier_score_loss(actual, probability)),
        })
    return pd.DataFrame(rows)


def economic_bucket_frame(frame, probability_column, dataset_name):
    work = frame.copy()
    work['policy_return'] = np.where(
        work['target_net3_within_5m'].eq(1), 0.03, work['timeout_net_return_5m']
    )
    work['probability_bucket'] = pd.qcut(
        work[probability_column].rank(method='first'), q=10, labels=False
    )
    result = work.groupby('probability_bucket').agg(
        samples=(probability_column, 'size'), mean_probability=(probability_column, 'mean'),
        tp_rate=('target_net3_within_5m', 'mean'),
        mean_timeout_return=('timeout_net_return_5m', 'mean'),
        mean_policy_return=('policy_return', 'mean'),
        mean_best_return=('best_min_bid_net_return_1_5m', 'mean'),
        mean_worst_return=('worst_min_bid_net_return_1_5m', 'mean'),
    ).reset_index()
    result['dataset'] = dataset_name
    return result


at_probability_metrics = pd.concat([
    at_probability_metric_frame(oof, 'oof'),
    at_probability_metric_frame(test_frame, 'test'),
], ignore_index=True)
economic_buckets = pd.concat([
    economic_bucket_frame(threshold_frame, p5_column, 'rolling_oof'),
    economic_bucket_frame(test_frame, TEST_P5_COLUMN, 'test'),
], ignore_index=True)

model_path = MODEL_ROOT / f'{MODEL_VERSION}.pt'
metrics_path = RESULT_ROOT / f'{MODEL_VERSION}_metrics.json'
fold_path = RESULT_ROOT / f'{MODEL_VERSION}_fold_metrics.parquet'
history_path = RESULT_ROOT / f'{MODEL_VERSION}_training_history.parquet'
oof_path = RESULT_ROOT / f'{MODEL_VERSION}_oof_predictions.parquet'
test_path = RESULT_ROOT / f'{MODEL_VERSION}_test_predictions.parquet'
threshold_path = RESULT_ROOT / f'{MODEL_VERSION}_threshold_candidates.parquet'
trade_path = RESULT_ROOT / f'{MODEL_VERSION}_test_trades.parquet'
probability_metric_path = RESULT_ROOT / f'{MODEL_VERSION}_probability_metrics.parquet'
reliability_path = RESULT_ROOT / f'{MODEL_VERSION}_reliability.parquet'
at_metric_path = RESULT_ROOT / f'{MODEL_VERSION}_at_probability_metrics.parquet'
economic_bucket_path = RESULT_ROOT / f'{MODEL_VERSION}_economic_probability_buckets.parquet'

checkpoint = {
    'model_version': MODEL_VERSION, 'architecture': 'MultiHorizonHazardTCN',
    'state_dict': {name: value.detach().cpu() for name, value in final_model.state_dict().items()},
    'input_dim': sequence.shape[-1], 'sequence_length': sequence.shape[1],
    'feature_names': schema['default_feature_names'], 'model_horizons': MODEL_HORIZONS,
    'scaler_center': torch.from_numpy(final_center.copy()),
    'scaler_scale': torch.from_numpy(final_scale.copy()),
    'platt_calibrators': FINAL_CALIBRATORS,
    'selected_probability_type': SELECTED_PROBABILITY_TYPE,
    'research_probability_threshold': SELECTED_THRESHOLD,
    'authorized_probability_threshold': SELECTED_THRESHOLD if DEPLOYMENT_ELIGIBLE else None,
    'final_epochs': FINAL_EPOCHS,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}
torch.save(checkpoint, model_path)
fold_metrics.to_parquet(fold_path, index=False, compression='zstd')
training_history.to_parquet(history_path, index=False, compression='zstd')
oof.to_parquet(oof_path, index=False, compression='zstd')
test_frame.to_parquet(test_path, index=False, compression='zstd')
threshold_candidates.to_parquet(threshold_path, index=False, compression='zstd')
test_trades.to_parquet(trade_path, index=False, compression='zstd')
pd.concat([
    oof_probability_metrics.assign(dataset='rolling_oof'),
    test_probability_metrics.assign(dataset='test'),
], ignore_index=True).to_parquet(probability_metric_path, index=False, compression='zstd')
reliability.to_parquet(reliability_path, index=False, compression='zstd')
at_probability_metrics.to_parquet(at_metric_path, index=False, compression='zstd')
economic_buckets.to_parquet(economic_bucket_path, index=False, compression='zstd')

def records_for_json(frame):
    return json.loads(frame.to_json(orient='records'))

metrics = {
    'model_version': MODEL_VERSION,
    'parameters': sum(parameter.numel() for parameter in final_model.parameters()),
    'model_horizons': MODEL_HORIZONS, 'oof_sessions': OOF_SESSIONS,
    'threshold_oof_sessions': OOF_SESSIONS[1:], 'test_session': TEST_SESSION,
    'best_epochs': best_epochs, 'final_epochs': FINAL_EPOCHS,
    'selected_probability_type': SELECTED_PROBABILITY_TYPE,
    'selected_probability_threshold': SELECTED_THRESHOLD,
    'oof_economic_eligible': OOF_ECONOMIC_ELIGIBLE,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
    'selected_oof_policy': {
        key: (None if pd.isna(value) else float(value)) for key, value in selected_policy.to_dict().items()
    },
    'test_policy': test_policy,
    'test_probability_metrics': records_for_json(test_probability_metrics),
    'at_probability_metrics': records_for_json(at_probability_metrics),
    'final_platt_calibrators': FINAL_CALIBRATORS,
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('model:', model_path)
print('metrics:', metrics_path)

model: /home/user/urbandatalab/YSLee/model/buy_multihorizon_hazard_tcn_v1.pt
metrics: /home/user/urbandatalab/YSLee/results/modeling/buy_multihorizon_hazard_tcn_v1_metrics.json
